## DESKRIPSI PROJECT 

"Klasifikasi Citra MRI Otak untuk Deteksi Tumor Menggunakan Ekstraksi Fitur Tekstur GLCM dan Metode KNN, SVM, dan Random Forest"

https://kaggle.com/datasets/navoneel/brain-mri-images-for-brain-tumor-detection

Percobaan (Baseline): Resize, Grayscale

Percobaan 1: Median Filter, Kernel Sharpening

Percobaan 2: Histogram Equalization, Closing (Morfologi)

Percobaan 3: Opening (Morfologi), Closing (Morfologi) 

proses
BASELINE (Pembanding):
Dataset Otak → Resize → Grayscale → GLCM → KNN/SVM/RF → Evaluasi

PERCOBAAN 1:
Dataset Otak → Resize → Grayscale → Median Filter → Kernel Sharpening → GLCM → KNN/SVM/RF → Evaluasi

PERCOBAAN 2:
Dataset Otak → Resize → Grayscale → Histogram Equalization → Closing → GLCM → KNN/SVM/RF → Evaluasi

PERCOBAAN 3:
Dataset Otak → Resize → Grayscale → Opening → Closing → GLCM → KNN/SVM/RF → Evaluasi

## IMPORT LIBRARY

In [2]:
# Import library yang kalian butuhkan
import os
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.metrics import accuracy_score, classification_report
from skimage.feature import graycomatrix, graycoprops
from scipy.stats import entropy
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay)

## Median Filter, Kernel Sharpening

In [ ]:
def filter_spasial(img, size, mode):
    if len(img.shape) == 3:
        height, width, channels = img.shape
        canvas_rgb = np.zeros_like(img, dtype=np.uint8)
        for c in range(channels):

            canvas_rgb[:, :, c] = filter_spasial(img[:, :, c], size, mode)
            
        return canvas_rgb
    
    height, width = img.shape
    pad = size // 2
    padded = np.pad(img, pad, mode='edge')
    canvas = np.zeros_like(img, dtype=np.uint8)

    match mode:
        case 'mean':
            area = size * size
            for i in range(height):
                for j in range(width):
                    # ambil region kernel
                    region = padded[i:i+size, j:j+size]

                    total = 0
                    for row in region:
                        for pixel in row:
                            total += int(pixel)
                    canvas[i, j] = total // area

        case 'median':
            for i in range(height):
                for j in range(width):

                    region = padded[i:i+size, j:j+size]

                    values = []
                    for row in region:
                        for pixel in row:
                            values.append(int(pixel))


                    n = len(values)
                    for a in range(n):
                        for b in range(0, n - a - 1):
                            if values[b] > values[b + 1]:
                                values[b], values[b + 1] = values[b + 1], values[b]

                    mid = n // 2
                    if n % 2 == 0:
                        canvas[i, j] = (values[mid - 1] + values[mid]) // 2
                    else:
                        canvas[i, j] = values[mid]

        case 'modus':
            for i in range(height):
                for j in range(width):
                    region = padded[i:i+size, j:j+size]
                    values = region.ravel()
                    count = {}
                    for val in values:
                        if val in count:
                            count[val] += 1
                        else:
                            count[val] = 1
                    max_count = 0
                    mode_val = 0
                    for val, freq in count.items():
                        if freq > max_count:
                            max_count = freq
                            mode_val = val
                    canvas[i, j] = mode_val

    return canvas






## kernel sharpening + convolution

In [ ]:
def convolution(img, kernel):
    if len(img.shape) == 3:
        height, width, channels = img.shape
        canvas_rgb = np.zeros_like(img).astype(np.float32)
        for c in range(channels):
            canvas_rgb[:, :, c] = convolution(img[:, :, c], kernel)
        return canvas_rgb

    size = kernel.shape[0]
    pad_size = size // 2
    padded = np.pad(img, pad_size, mode='constant')
    canvas = np.zeros_like(img).astype(np.float32)
    height, width = img.shape
    
    for i in range(height):
        for j in range(width):
            region = padded[i:i+size, j:j+size]
            canvas[i, j] = np.sum(region * kernel)
            
    return canvas
  
  
  # kernel penajam gambar (sharpening)
kernelSharpening = np.array([
    [ 0, -1,  0 ],
    [-1,  5, -1 ],
    [ 0, -1,  0 ]
])



## Histogram Equalization

In [ ]:
def hitung_histogram(gambar):

    img_arr = np.array(gambar).astype(float)

    # grayscale luminance
    if len(img_arr.shape) == 3:
        img_arr = (
            0.299 * img_arr[:,:,0] +
            0.587 * img_arr[:,:,1] +
            0.114 * img_arr[:,:,2]
        )

    if img_arr.max() <= 1.1:
        img_arr = img_arr * 255

    piksel_flat = img_arr.flatten()

    histogram = [0] * 256

    for p in piksel_flat:

        p = int(round(p))

        if 0 <= p <= 255:
            histogram[p] += 1

    return histogram

## Closing + Opening

In [ ]:
# Thresholding: mengubah citra grayscale menjadi citra biner (0 atau 255)
def thresholding(img, batas):
    baris, kolom = img.shape
    canvas = np.zeros_like(img, dtype=np.uint8)
    for i in range(baris):
        for j in range(kolom):
            if img[i, j] > batas:
                canvas[i, j] = 255
            else:
                canvas[i, j] = 0
    return canvas


# Resize: mengubah ukuran citra menggunakan nearest-neighbor interpolation
def resize(image, new_width=256, new_height=256):
    old_height, old_width = image.shape[:2]
    resized = np.zeros((new_height, new_width), dtype=np.uint8)
    for i in range(new_height):
        for j in range(new_width):
            x = int(j * old_width / new_width)
            y = int(i * old_height / new_height)
            resized[i, j] = image[y, x]
    return resized


# Dilasi: memperluas area objek putih pada citra biner
def dilasi(image, kernel):
    height, width = image.shape
    k_height, k_width = kernel.shape
    center = k_height // 2
    hasil = np.zeros((height, width))
    for i in range(center, height - center):
        for j in range(center, width - center):
            if image[i, j] == 255:
                for k in range(k_height):
                    for l in range(k_width):
                        if kernel[k, l] == 1:
                            hasil[i + k - center, j + l - center] = 255
            else:
                if hasil[i, j] != 255:
                    hasil[i, j] = 0
    return hasil.astype(np.uint8)


# Erosi: menipiskan area objek putih pada citra biner
def erosi(image, kernel):
    height, width = image.shape
    k_height, k_width = kernel.shape
    center = k_height // 2
    hasil = np.zeros((height, width), dtype=np.uint8)
    for i in range(center, height - center):
        for j in range(center, width - center):
            cocok = True
            for k in range(k_height):
                for l in range(k_width):
                    if kernel[k, l] == 1 and image[i + k - center, j + l - center] == 0:
                        cocok = False
                        break
                if not cocok:
                    break
            if cocok:
                hasil[i, j] = 255
    return hasil


# Closing: dilasi diikuti erosi
def closing(image, kernel):
    return erosi(dilasi(image, kernel), kernel)
  
  
# Opening: erosi diikuti dilasi
def opening(image, kernel):
    return dilasi(erosi(image, kernel), kernel)

